# 12. Defensive-zone recovery pace and advancement


### Purpose

Assess whether 5v5 possessions starting with a defensive-zone puck recovery reach a controlled offensive-zone entry or lose possession first. The coaching question is: **can a team advance under control at higher post-recovery pace without losing the puck?**

### Expected outcome

One record per recovery-started possession, pooled slower/middle/faster pace bands, controlled-entry and possession-loss comparisons, and team and sequence reviews. Keep dumped entries, restarts, manpower changes, period endings, and unresolved endings separate from confirmed possession loss.


## 12.1. Setup and load processed data

Reuse augmented events, transition pace, and reference tables. Pool sources, retain provenance, and include only sequences beginning at 5v5. Possession ownership is inferred and modeled movement times are estimated.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_columns", None)
ROOT_DIR = Path("..")
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

pace_manifest = json.loads(
    (PROCESSED_DATA_DIR / "pace_export_manifest.json").read_text(encoding="utf-8")
)
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values("event_id")
transitions = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
teams = pd.read_parquet(PROCESSED_DATA_DIR / "teams.parquet")
games = pd.read_parquet(PROCESSED_DATA_DIR / "games.parquet")
players = pd.read_parquet(PROCESSED_DATA_DIR / "players.parquet")

display(pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in {"events": events, "transitions": transitions,
                        "teams": teams, "games": games, "players": players}.items()
]))
print("Timing:", pace_manifest["timing"])


,table,rows,columns
0,events,86670,27
1,transitions,63298,25
2,teams,20,3
3,games,34,5
4,players,435,4


Timing: Recorded clocks preserved; speeds use notebook 02 modeled clocks. Fractions are estimated, not measured.


## 12.2. Select defensive-zone recovery starts

Require the first event of a possession sequence to be `Puck Recovery`, credited to its owner, in the defensive zone (`x < 75`) at 5v5. Recoveries partway through an existing possession are excluded.

Retain the immediately following sequence's first event to assess how a possession ended. Never follow a recovery into a later game or period.


In [2]:
sequence_starts = events.groupby("possession_id", sort=False).head(1).copy()
sequence_starts["next_possession_id"] = sequence_starts["possession_id"].shift(-1)
recoveries = sequence_starts.loc[
    sequence_starts["event"].eq("Puck Recovery")
    & sequence_starts["is_5v5"]
    & sequence_starts["event_team_zone"].eq("DZ")
    & sequence_starts["team_id"].eq(sequence_starts["possession_team_id"])
].copy()
recoveries["team_name"] = recoveries["team_id"].map(teams.set_index("team_id")["team_name"])
recoveries["player_name"] = recoveries["player_id"].map(players.set_index("player_id")["player_name"])
recoveries["game_date"] = recoveries["game_id"].map(games.set_index("game_id")["game_date"])
sequence_events = {pid: group for pid, group in events.groupby("possession_id", sort=False)}
start_lookup = sequence_starts.set_index("possession_id")
print("Qualifying defensive-zone recovery possessions:", len(recoveries))


Qualifying defensive-zone recovery possessions: 5839


## 12.3. Classify the first outcome

A `Carried` or `Played` zone entry by the recovering team is a controlled entry. A `Dumped` entry is a separate outcome, not controlled advancement or confirmed loss. Stop at the first entry rather than following play after it.

If the sequence has no entry or goal, inspect the first event of the immediately following sequence. A faceoff or penalty-shot restart is a restart; a change away from 5v5 is a manpower ending. Otherwise an opponent control event confirms possession loss. Penalties and generated incomplete receptions alone do not establish control. Endings without this evidence remain unresolved. Game/period boundaries are separate endings.

This is an event-based classification: unrecorded stoppages and delayed penalty tagging can affect it. The two primary outcomes are mutually exclusive first outcomes; other endings remain in their denominator and are reported separately.


In [3]:
CONTROL_EVENTS = {"Puck Recovery", "Takeaway", "Pass", "Pass Reception", "Incomplete Pass",
                  "Shot", "Goal", "Dump In/Out", "Zone Entry"}
outcome_rows = []
for recovery in recoveries.itertuples(index=False):
    sequence = sequence_events[recovery.possession_id]
    later = sequence.loc[sequence["event_id"].gt(recovery.event_id)]
    outcome = None
    outcome_event_id = None
    # Without a focal action, keep all observed movements in this sequence.
    cutoff_id = int(sequence["event_id"].max()) + 1
    for event in later.itertuples(index=False):
        if event.team_id == recovery.team_id and event.event == "Zone Entry":
            outcome = "Controlled entry" if event.detail_1 in {"Carried", "Played"} else "Dumped entry"
            outcome_event_id = event.event_id
            cutoff_id = event.event_id
            break
        if event.event in {"Goal", "Penalty Shot Goal"}:
            outcome = "Goal / special play"
            outcome_event_id = event.event_id
            cutoff_id = event.event_id
            break
    if outcome is None:
        if pd.isna(recovery.next_possession_id):
            outcome = "Period / game ending"
        else:
            next_event = start_lookup.loc[int(recovery.next_possession_id)]
            if next_event["game_id"] != recovery.game_id or next_event["period"] != recovery.period:
                outcome = "Period / game ending"
            elif next_event["event"] in {"Faceoff Win", "Penalty Shot Goal"}:
                outcome = "Restart"
                outcome_event_id = int(next_event["event_id"])
            elif not next_event["is_5v5"]:
                outcome = "Manpower change"
                outcome_event_id = int(next_event["event_id"])
            elif next_event["team_id"] == recovery.opponent_team_id and next_event["event"] in CONTROL_EVENTS:
                outcome = "Possession loss"
                outcome_event_id = int(next_event["event_id"])
            else:
                outcome = "Unresolved ending"
    outcome_rows.append({
        "event_id": recovery.event_id, "outcome": outcome,
        "outcome_event_id": outcome_event_id, "pace_cutoff_event_id": cutoff_id,
        "controlled_entry": outcome == "Controlled entry", "possession_loss": outcome == "Possession loss",
        "sequence_span_seconds": float(recovery.modeled_clock_seconds - sequence["modeled_clock_seconds"].min()),
    })
recoveries = recoveries.merge(pd.DataFrame(outcome_rows), on="event_id", validate="one_to_one")
display(recoveries.groupby("outcome").size().rename("possessions").reset_index())


,outcome,possessions
0,Controlled entry,1198
1,Dumped entry,478
2,Goal / special play,2
3,Manpower change,6
4,Period / game ending,23
5,Possession loss,3661
6,Restart,471


## 12.4. Calculate pace after recovery, before the outcome

Pool eligible movement after recovery within the same sequence. For entries/goals, require end event IDs strictly before the focal action, excluding movement ending at it. For opponent possession loss or another sequence ending, retain all eligible movement within the recovery sequence and never bridge to the next owner.

Empty and zero-duration movement histories have unknown pace and remain outside tertiles, with coverage reported. This follows the retrospective convention in notebook 11: outcome-dependent measurement windows differ. Faster pace associated with an outcome is not evidence that instructing players to speed up would produce it.


In [4]:
valid_movements = transitions.loc[transitions["exclusion_reason"].isna()]
movement_groups = {
    pid: group[["start_event_id", "end_event_id", "distance_ft", "modeled_elapsed_seconds"]].to_numpy(float)
    for pid, group in valid_movements.groupby("possession_id", sort=False)
}
pace_rows = []
for recovery in recoveries.itertuples(index=False):
    movement = movement_groups.get(recovery.possession_id)
    if movement is not None:
        movement = movement[(movement[:, 0] >= recovery.event_id)
                            & (movement[:, 1] < recovery.pace_cutoff_event_id)]
    count = 0 if movement is None else len(movement)
    distance = 0.0 if movement is None else float(movement[:, 2].sum())
    duration = 0.0 if movement is None else float(movement[:, 3].sum())
    pace_rows.append({
        "event_id": recovery.event_id, "movement_transitions": count,
        "movement_distance_ft": distance, "movement_seconds": duration,
        "post_recovery_pace_ft_s": distance / duration if duration > 0 else np.nan,
        "pace_status": "eligible" if duration > 0 else "no_movement" if count == 0 else "zero_duration",
    })
recoveries = recoveries.merge(pd.DataFrame(pace_rows), on="event_id", validate="one_to_one")
coverage = recoveries.groupby(["pace_status", "outcome"]).size().rename("possessions").reset_index()
display(coverage)


,pace_status,outcome,possessions
0,eligible,Controlled entry,1073
1,eligible,Dumped entry,477
2,eligible,Goal / special play,2
3,eligible,Manpower change,5
4,eligible,Period / game ending,10
5,eligible,Possession loss,2742
6,eligible,Restart,170
7,no_movement,Controlled entry,124
8,no_movement,Manpower change,1
9,no_movement,Period / game ending,12


## 12.5. Define pooled pace tertiles and compare outcomes

Share slower/middle/faster boundaries across teams using all recovery possessions with measurable post-recovery pace, including other endings. Show the controlled-entry and confirmed-loss percentages as separate bars without uncertainty intervals. Both use all measurable recovery possessions in their band as denominator, so they need not sum to 100%.

Report the complete outcome breakdown beside the primary chart. Dumped entries and censored/unresolved endings must not be silently converted to losses or removed to inflate advancement rates. Retain sequence spans and game counts when interpreting differences.


In [5]:
analysis_recoveries = recoveries.loc[recoveries["pace_status"].eq("eligible")].copy()
bins, pace_edges = pd.qcut(analysis_recoveries["post_recovery_pace_ft_s"], q=3,
                          labels=False, retbins=True, duplicates="drop")
band_names = {1:"Slower", 2:"Middle", 3:"Faster"}
analysis_recoveries["pace_tertile"] = (bins + 1).astype("Int64")
analysis_recoveries["pace_band"] = analysis_recoveries["pace_tertile"].map(band_names)
pace_boundaries = pd.DataFrame({"pace_tertile":np.arange(1,len(pace_edges)),
    "lower_pace_ft_s":pace_edges[:-1], "upper_pace_ft_s":pace_edges[1:]})
pace_boundaries["pace_band"] = pace_boundaries["pace_tertile"].map(band_names)
display(pace_boundaries.round(2))

recovery_summary = analysis_recoveries.groupby("pace_band").agg(
    possessions=("possession_id","size"), games=("game_id","nunique"),
    controlled_entries=("controlled_entry","sum"), possession_losses=("possession_loss","sum"),
    median_pace_ft_s=("post_recovery_pace_ft_s","median"),
    median_sequence_span_seconds=("sequence_span_seconds","median"),
).reset_index()
recovery_summary["Controlled entry"] = 100 * recovery_summary["controlled_entries"] / recovery_summary["possessions"]
recovery_summary["Possession loss"] = 100 * recovery_summary["possession_losses"] / recovery_summary["possessions"]
display(recovery_summary.round(2))
outcome_breakdown = analysis_recoveries.groupby(["pace_band","outcome"]).size().rename("possessions").reset_index()
outcome_breakdown["outcome_pct"] = 100 * outcome_breakdown["possessions"] / outcome_breakdown.groupby("pace_band")["possessions"].transform("sum")
display(outcome_breakdown.round(2))

plot_data = recovery_summary.melt(
    id_vars=["pace_band","possessions","games","median_pace_ft_s","median_sequence_span_seconds"],
    value_vars=["Controlled entry","Possession loss"], var_name="outcome", value_name="outcome_pct",
)
recovery_fig = px.bar(plot_data,x="pace_band",y="outcome_pct",color="pace_band",facet_col="outcome",
    category_orders={"pace_band":["Slower","Middle","Faster"],"outcome":["Controlled entry","Possession loss"]},
    color_discrete_map={"Slower":"#93c5fd","Middle":"#3b82f6","Faster":"#1d4ed8"},
    hover_data=["possessions","games","median_pace_ft_s","median_sequence_span_seconds"],
    labels={"pace_band":"Post-recovery sequence pace","outcome_pct":"Recovery possessions (%)"},
    title="5v5 defensive-zone recovery outcomes | retrospective comparison")
recovery_fig.update_layout(template="plotly_white",height=450,showlegend=False)
recovery_fig.update_yaxes(range=[0,100])
recovery_fig.show()


,pace_tertile,lower_pace_ft_s,upper_pace_ft_s,pace_band
0,1,0.00,14.58,Slower
1,2,14.58,22.18,Middle
2,3,22.18,372.00,Faster


,pace_band,possessions,games,controlled_entries,possession_losses,median_pace_ft_s,median_sequence_span_seconds,Controlled entry,Possession loss
0,Faster,1493,34,669,604,28.40,5.0,44.81,40.46
1,Middle,1492,34,305,892,18.25,6.0,20.44,59.79
2,Slower,1494,34,99,1246,9.99,3.0,6.63,83.4


,pace_band,outcome,possessions,outcome_pct
0,Faster,Controlled entry,669,44.81
1,Faster,Dumped entry,162,10.85
2,Faster,Goal / special play,2,0.13
3,Faster,Manpower change,1,0.07
4,Faster,Possession loss,604,40.46
5,Faster,Restart,55,3.68
6,Middle,Controlled entry,305,20.44
7,Middle,Dumped entry,235,15.75
8,Middle,Manpower change,2,0.13
9,Middle,Possession loss,892,59.79


## 12.6. Review teams and individual possessions

Retain sample sizes and other endings when comparing team profiles. Review actual recovery locations, passing support, and opponent control evidence. Unknown pace includes immediate turnovers and actions, and its exclusion can materially affect the chart.

Before making recommendations, assess consistency across games, inferred boundaries, and modeled timing. A fixed initial pace window followed by a separate outcome window would be needed for a predictive test.


In [6]:
team_summary = analysis_recoveries.groupby(["team_id","team_name","pace_band"]).agg(
    possessions=("possession_id","size"),games=("game_id","nunique"),
    controlled_entry_rate=("controlled_entry","mean"),possession_loss_rate=("possession_loss","mean"),
).reset_index()
display(team_summary.round(3))
REVIEW_PACE_BAND = "Faster"
REVIEW_OUTCOME = "Possession loss"
review_recoveries = analysis_recoveries.loc[
    analysis_recoveries["pace_band"].eq(REVIEW_PACE_BAND)
    & analysis_recoveries["outcome"].eq(REVIEW_OUTCOME)
]
display(review_recoveries[["game_id","game_date","team_name","player_name","period","clock_seconds",
    "event_id","possession_id","x","y","post_recovery_pace_ft_s","movement_seconds",
    "movement_transitions","sequence_span_seconds","outcome","outcome_event_id"]].head(20))


,team_id,team_name,pace_band,possessions,games,controlled_entry_rate,possession_loss_rate
0,1,Boston Pride,Faster,162,7,0.475,0.420
1,1,Boston Pride,Middle,137,7,0.241,0.577
2,1,Boston Pride,Slower,118,7,0.059,0.847
3,2,Buffalo Beauts,Faster,118,6,0.441,0.407
4,2,Buffalo Beauts,Middle,135,6,0.170,0.630
5,2,Buffalo Beauts,Slower,147,6,0.048,0.844
6,3,Connecticut Whale,Faster,94,4,0.447,0.404
7,3,Connecticut Whale,Middle,101,4,0.198,0.634
8,3,Connecticut Whale,Slower,97,4,0.031,0.897
9,4,Metropolitan Riveters,Faster,74,3,0.432,0.473


,game_id,game_date,team_name,player_name,period,clock_seconds,event_id,possession_id,x,y,post_recovery_pace_ft_s,movement_seconds,movement_transitions,sequence_span_seconds,outcome,outcome_event_id
2,3,2021-01-23,Minnesota Whitecaps,Allie Thunstrom,1,1129.0,46,11,39.0,45.0,23.923694,8.0,7,8.0,Possession loss,55.0
6,3,2021-01-23,Minnesota Whitecaps,Stephanie Anderson,1,1055.0,104,22,37.0,80.0,28.808438,3.0,3,3.0,Possession loss,109.0
13,3,2021-01-23,Boston Pride,Jenna Rheault,1,947.0,182,34,4.0,45.0,22.463439,4.0,5,4.0,Possession loss,188.0
28,3,2021-01-23,Boston Pride,Paige Capistran,1,613.0,401,77,14.0,37.0,28.861739,1.0,1,1.0,Possession loss,404.0
76,3,2021-01-23,Boston Pride,Kaleigh Fratkin,2,637.0,1104,218,38.0,41.0,26.484431,6.0,2,6.0,Possession loss,1108.0
91,3,2021-01-23,Boston Pride,Mallory Souliotis,2,341.0,1297,248,0.0,39.0,53.037722,1.0,3,1.0,Possession loss,1302.0
102,3,2021-01-23,Minnesota Whitecaps,Emma Stauber,3,1113.0,1546,288,0.0,50.0,23.631805,3.0,3,3.0,Possession loss,1551.0
103,3,2021-01-23,Minnesota Whitecaps,Haylea Schmid,3,1092.0,1561,293,23.0,1.0,23.693270,8.0,6,8.0,Possession loss,1568.0
108,3,2021-01-23,Minnesota Whitecaps,Sydney Baldwin,3,1001.0,1620,303,19.0,48.0,24.791113,2.0,3,2.0,Possession loss,1625.0
123,3,2021-01-23,Minnesota Whitecaps,Haley Mack,3,733.0,1793,332,60.0,8.0,24.083189,0.5,2,0.5,Possession loss,1796.0
